***Problem Statement*** : Hospital Patient Data Analysis
Context:
A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.


In [1]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

Saving Patient_Data.csv to Patient_Data.csv
Saving Billing_Data.csv to Billing_Data.csv


1.	Load the patient dataset and show summary with info().

In [2]:
df = pd.read_csv('Patient_Data.csv')
df_billing = pd.read_csv('Billing_Data.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [24]:
df_patient = df.copy()
print(df_patient)

   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John         NaN               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose         NaN               2   
5        101    Alice   Cardiology  Dr. Smith      5000.0               1   

        CheckInTime  
0  2023-01-10 09:00  
1  2023-01-11 10:30  
2  2023-01-12 11:00  
3  2023-01-13 12:00  
4  2023-01-14 08:45  
5  2023-01-10 09:00  


3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [25]:
df_patient.drop(['ReceptionistID','CheckInTime'],inplace = True,axis = 1)

df_patient


,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN
5,101,Alice,Cardiology,Dr. Smith,5000.0


4.	Use groupby to find total bill amount per department.

In [5]:
df_patient['BillAmount'].groupby(df_patient['Department']).sum()

,BillAmount
Department,
Cardiology,16200.0
Dermatology,0.0
Neurology,0.0
Orthopedics,7500.0


5.	Remove duplicate patient records based on PatientID.

In [26]:
print(df_patient['PatientID'].groupby(df_patient['PatientID']).count())
print(f"\n {df_patient['PatientID'].nunique()} Unique Patients Ids are : {df_patient['PatientID'].unique()}")

PatientID
101    2
102    1
103    1
104    1
105    1
Name: PatientID, dtype: int64

 5 Unique Patients Ids are : [101 102 103 104 105]


In [27]:
df_patient.drop_duplicates(subset=['PatientID'], keep='first',inplace = True)
print(f"After dropping duplicates: \n{df_patient['PatientID']}")

After dropping duplicates: 
0    101
1    102
2    103
3    104
4    105
Name: PatientID, dtype: int64


6.	Fill missing BillAmount values with the mean bill amount.

In [28]:
print(df_patient['BillAmount'])

0    5000.0
1       NaN
2    7500.0
3    6200.0
4       NaN
Name: BillAmount, dtype: float64


In [29]:
df_patient['BillAmount'].fillna(df_patient['BillAmount'].mean(),inplace = True)

print(df_patient['BillAmount'])


0    5000.000000
1    6233.333333
2    7500.000000
3    6200.000000
4    6233.333333
Name: BillAmount, dtype: float64


/tmp/ipython-input-738110873.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_patient['BillAmount'].fillna(df_patient['BillAmount'].mean(),inplace = True)


7.	Merge the billing dataset with patient dataset on PatientID.

In [30]:
print(f"Billing: \n{df_billing}")

print(f"\nPatient: \n{df_patient}")

Billing: 
   PatientID  InsuranceCovered  FinalAmount
0        101              2000         3000
1        102              1500         3500
2        103              2500         5000
3        104              3000         3200
4        105              1000         4000

Patient: 
   PatientID     Name   Department     Doctor   BillAmount
0        101    Alice   Cardiology  Dr. Smith  5000.000000
1        102      Bob    Neurology   Dr. John  6233.333333
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000
3        104    David   Cardiology  Dr. Smith  6200.000000
4        105      Eva  Dermatology   Dr. Rose  6233.333333


In [31]:
df_patient_bill = pd.merge(df_patient,df_billing,on='PatientID')

print(f"Merged Data: \n{df_patient_bill}")

Merged Data: 
   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              2000   
1        102      Bob    Neurology   Dr. John  6233.333333              1500   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              2500   
3        104    David   Cardiology  Dr. Smith  6200.000000              3000   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


8.	Concatenate an additional DataFrame that contains new patients for the current week

In [33]:
df_new_patients = pd.DataFrame(
    {
        "PatientID": [106,107,108,109],
        "Name": ["Alina", "Manoj", "Ganesh","Esther"],
        "Department": ["Dermatology", "Orthopedics", "Neurology", "Cardiology"],
        "Doctor": ["Dr. Rose", "Dr. Lee", "Dr. John", "Dr. Smith"],
        "BillAmount":[5000,2000,1500,3000],
    }
)
df_new_patients


,PatientID,Name,Department,Doctor,BillAmount
0,106,Alina,Dermatology,Dr. Rose,5000
1,107,Manoj,Orthopedics,Dr. Lee,2000
2,108,Ganesh,Neurology,Dr. John,1500
3,109,Esther,Cardiology,Dr. Smith,3000


9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [34]:
df_new_columns = pd.DataFrame(
    {"InsuranceCovered":[1000,2000,1500,3000],
     "FinalAmount":[2000,1500,2000,1000]})
df_new_columns

,InsuranceCovered,FinalAmount
0,1000,2000
1,2000,1500
2,1500,2000
3,3000,1000


In [35]:
df_new_patients = pd.concat([df_new_patients,df_new_columns],axis=1)

df_new_patients



,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,106,Alina,Dermatology,Dr. Rose,5000,1000,2000
1,107,Manoj,Orthopedics,Dr. Lee,2000,2000,1500
2,108,Ganesh,Neurology,Dr. John,1500,1500,2000
3,109,Esther,Cardiology,Dr. Smith,3000,3000,1000


In [36]:
df_patient_bill = pd.concat([df_patient_bill,df_new_patients],ignore_index=True,axis=0)

df_patient_bill

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000
5,106,Alina,Dermatology,Dr. Rose,5000.000000,1000,2000
6,107,Manoj,Orthopedics,Dr. Lee,2000.000000,2000,1500
7,108,Ganesh,Neurology,Dr. John,1500.000000,1500,2000
8,109,Esther,Cardiology,Dr. Smith,3000.000000,3000,1000
